# Embedding using ResNet50

Performed embedding of blur images (that has been done in the last step).

In [1]:
import cv2
import os
import numpy as np
import tensorflow as tf
import joblib
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.models import Model
from pathlib import Path

In [2]:

TUNING_CONFIG = {

    "input_mfrat_dir": Path(r"C:\Users\singh\Users\singh\Desktop\Palm detection\ROI\MFRT image new"),
    "output_embedding_path": Path(r"C:\Users\singh\Users\singh\Desktop\Palm detection\ROI\05_palm_embeddings latest work.pkl"),
    
  
    "target_resolution": (224, 224)
}


Loading ResNet50 to perform embedding

In [3]:
# Initializing  pre-trained ResNet50 model 

print("Loading pre-trained ResNet50...")
# Load ResNet50 without the final classification head
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
# Add Global Average Pooling to flatten the feature maps into a 2048-dimensional vector
pooling_layer = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
feature_extractor = Model(inputs=base_model.input, outputs=pooling_layer)


Loading pre-trained ResNet50...


In [4]:


if __name__ == "__main__":
    # Scan the MFRAT folder for all processed image maps recursively
    mfrat_images = list(TUNING_CONFIG["input_mfrat_dir"].rglob('*.png'))
    
    if not mfrat_images:
        print(f"No images found in {TUNING_CONFIG['input_mfrat_dir']}. Ensure Step 6 ran successfully!")
    else:
        print(f"Starting separate vector embedding extraction for {len(mfrat_images)} items...")
        
        # Dictionaries to store database metadata
        embeddings_db = []
        labels_db = []
        filepaths_db = []
        handedness_db = []
        
        processed_count = 0
        
        for img_path in mfrat_images:
            # 1. Load the MFRAT image as Grayscale (1-channel)
            mfrat_img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            if mfrat_img is None: 
                continue
            
            # Ensure it strictly matches the Target Size
            if mfrat_img.shape != TUNING_CONFIG["target_resolution"]:
                mfrat_img = cv2.resize(mfrat_img, TUNING_CONFIG["target_resolution"])
                
            # 2. CONVERT TO 3-CHANNEL: Map 1-Ch gray to 3-Ch RGB for ResNet compatibility
            img_rgb = cv2.cvtColor(mfrat_img, cv2.COLOR_GRAY2RGB)
            
            # 3. PREPARE FOR DEEP LEARNING: Add batch dimension and scale inputs via ResNet rules
            img_batch = np.expand_dims(img_rgb, axis=0).astype(np.float32)
            preprocessed_batch = preprocess_input(img_batch)
            
            # 4. RUN INFERENCE: Extract the 2048-dimensional vector
            features = feature_extractor.predict(preprocessed_batch, verbose=0)[0]
            
            # 5. L2 NORMALIZATION: Force unit sphere scaling for stable SVM decision boundaries
            normalized_features = features / np.linalg.norm(features)
            
            # Parse structural metadata directly from your clean folder layout
            # Path layout: input_mfrat_dir / User_ID / Handedness (Left/Right) / filename.png
            hand_label = img_path.parent.name         # "Left" or "Right"
            user_id = img_path.parent.parent.name     # e.g., "pranav", "lakshay"
            
            # Append metadata to database storage matrices
            embeddings_db.append(normalized_features)
            labels_db.append(user_id)
            handedness_db.append(hand_label)
            filepaths_db.append(str(img_path))
            
            processed_count += 1
            if processed_count % 50 == 0:
                print(f"   Processed {processed_count}/{len(mfrat_images)} images...")
                
        # Bundle all data vectors neatly into a single dictionary object
        dataset_package = {
            "embeddings": np.array(embeddings_db, dtype=np.float32),
            "labels": labels_db,
            "handedness": handedness_db,
            "filepaths": filepaths_db
        }
        
        # Save output dictionary package into a single serialization file
        output_file = TUNING_CONFIG["output_embedding_path"]
        os.makedirs(output_file.parent, exist_ok=True)
        joblib.dump(dataset_package, str(output_file))
        
        print("=" * 55)
        print(f"Stage 7 Complete. Feature embedding generation finished.")
        print(f"Total extracted feature profiles saved: {processed_count}")
        print(f"Output Matrix Dimensions: {dataset_package['embeddings'].shape}")
        print(f"Saved Database File Path: {output_file}")
        print("=" * 55)

Starting separate vector embedding extraction for 2288 items...
   Processed 50/2288 images...
   Processed 100/2288 images...
   Processed 150/2288 images...
   Processed 200/2288 images...
   Processed 250/2288 images...
   Processed 300/2288 images...
   Processed 350/2288 images...
   Processed 400/2288 images...
   Processed 450/2288 images...
   Processed 500/2288 images...
   Processed 550/2288 images...
   Processed 600/2288 images...
   Processed 650/2288 images...
   Processed 700/2288 images...
   Processed 750/2288 images...
   Processed 800/2288 images...
   Processed 850/2288 images...
   Processed 900/2288 images...
   Processed 950/2288 images...
   Processed 1000/2288 images...
   Processed 1050/2288 images...
   Processed 1100/2288 images...
   Processed 1150/2288 images...
   Processed 1200/2288 images...
   Processed 1250/2288 images...
   Processed 1300/2288 images...
   Processed 1350/2288 images...
   Processed 1400/2288 images...
   Processed 1450/2288 images...